# HanuAi ML Assessment — Task 1: Web Scraping & Sentiment Analysis

**Objective:** Scrape product reviews from BestBuy Canada, perform sentiment analysis, and categorize reviews into aspect-based sentiment tags.

**Product Selected:** Sony WH-1000XM5 Wireless Noise Cancelling Headphones  
**URL:** https://www.bestbuy.ca/en-ca/product/sony-wh-1000xm5-wireless-noise-cancelling-headphones-black/16191426  
**Review Count:** 200+ reviews  

---

## Notebook Structure
1. Setup & Imports  
2. Web Scraper (Playwright + BeautifulSoup)  
3. Data Cleaning & Export  
4. Sentiment Analysis (VADER + Zero-Shot Classification)  
5. Aspect-Based Categorization  
6. Business Insights  
7. Anti-Scraping Mitigation Strategies  

## Section 1 — Setup & Imports

Install all required libraries. Playwright handles dynamic JS rendering; BeautifulSoup parses HTML; VADER and Transformers provide sentiment scoring.

In [ ]:
# Install dependencies (run once)
# Playwright: headless browser automation for JS-rendered pages
# vaderSentiment: fast rule-based sentiment lexicon (no GPU needed)
# transformers: HuggingFace zero-shot classification for aspect tagging
# openpyxl: Excel export

import subprocess, sys

packages = [
    'playwright', 'beautifulsoup4', 'vaderSentiment',
    'transformers', 'torch', 'pandas', 'openpyxl',
    'tqdm', 'fake-useragent'
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet'] + packages)

# Install Playwright browser binaries (Chromium)
subprocess.run(['playwright', 'install', 'chromium'])
print('All dependencies installed.')

In [ ]:
# Core Imports
import asyncio
import re
import time
import random
import uuid
import json
from datetime import datetime
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm

# Playwright async API
from playwright.async_api import async_playwright, TimeoutError as PWTimeoutError

# Sentiment analysis
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

print('Imports successful.')

## Section 2 — Web Scraper

### Design Overview
BestBuy Canada renders reviews via JavaScript (Bazaarvoice API calls). We use **Playwright** to:
1. Load the product page in a headless Chromium browser  
2. Apply sort filters (Most Relevant, Most Helpful, Newest, Highest, Lowest)  
3. Click 'Load More' / paginate until all reviews are visible  
4. Extract raw HTML parsed with **BeautifulSoup**  

### Anti-Detection Measures Applied
- Random `user-agent` strings via `fake_useragent`  
- Randomised human-like delays between actions  
- Stealth mode (disables `navigator.webdriver` flag)  
- Viewport randomisation  

In [ ]:
# Configuration
# Target product URL: Sony WH-1000XM5 headphones (200+ reviews)
PRODUCT_URL = (
    'https://www.bestbuy.ca/en-ca/product/'
    'sony-wh-1000xm5-wireless-noise-cancelling-headphones-black/16191426'
)

SOURCE_PLATFORM = 'BestBuy Canada'
OUTPUT_CSV  = 'bestbuy_reviews_raw.csv'
OUTPUT_XLSX = 'bestbuy_reviews_sentiment.xlsx'

# Sort filter labels as they appear in the BestBuy UI
SORT_FILTERS = [
    'Most Relevant',
    'Most Helpful',
    'Newest',
    'Highest Rating',
    'Lowest Rating',
]

# Human-like delay range (seconds) between actions to reduce bot detection risk
MIN_DELAY, MAX_DELAY = 1.5, 3.5

def human_delay(min_s=MIN_DELAY, max_s=MAX_DELAY):
    """Introduce a randomised pause to mimic human browsing behaviour."""
    time.sleep(random.uniform(min_s, max_s))

In [ ]:
# HTML Parser: extract reviews from a page snapshot
def parse_reviews_from_html(html: str, filter_name: str) -> list:
    """
    Parse individual review blocks from the rendered HTML.

    BestBuy Canada uses Bazaarvoice components. Each review lives inside
    a container identified by data-automation attributes.

    Args:
        html        : Full page HTML string from Playwright.
        filter_name : Name of the active sort filter (for provenance tracking).

    Returns:
        List of dicts, one per review.
    """
    soup = BeautifulSoup(html, 'html.parser')
    reviews = []

    # BestBuy Canada review containers (selector stable as of Oct 2024)
    # Each review card is inside a div with data-automation='review-item'
    review_cards = soup.find_all('div', attrs={'data-automation': 'review-item'})

    # Fallback: BV (Bazaarvoice) rendered structure
    if not review_cards:
        review_cards = soup.find_all('li', class_=re.compile(r'bv-content-item'))

    for card in review_cards:
        try:
            # Reviewer Name
            name_el = (
                card.find('div', attrs={'data-automation': 'review-author'}) or
                card.find('span', class_=re.compile(r'bv-author-name'))
            )
            reviewer_name = name_el.get_text(strip=True) if name_el else 'Anonymous'

            # Review Title
            title_el = (
                card.find('div', attrs={'data-automation': 'review-title'}) or
                card.find('h3', class_=re.compile(r'bv-content-title'))
            )
            title = title_el.get_text(strip=True) if title_el else ''

            # Review Body
            body_el = (
                card.find('div', attrs={'data-automation': 'review-text'}) or
                card.find('div', class_=re.compile(r'bv-content-summary-body-text'))
            )
            review_text = body_el.get_text(strip=True) if body_el else ''

            # Star Rating — typically in aria-label like 'Rated 4 out of 5 stars'
            rating = None
            rating_el = card.find(attrs={'aria-label': re.compile(r'Rated \d out of 5')})
            if rating_el:
                m = re.search(r'Rated (\d)', rating_el['aria-label'])
                rating = int(m.group(1)) if m else None
            else:
                star_el = card.find(attrs={'data-automation': 'rating'})
                if star_el and star_el.get('aria-label'):
                    m = re.search(r'(\d)', star_el['aria-label'])
                    rating = int(m.group(1)) if m else None

            # Date — normalise to YYYY-MM-DD
            date_el = (
                card.find('time') or
                card.find('span', attrs={'data-automation': 'review-date'})
            )
            raw_date = date_el.get('datetime', date_el.get_text(strip=True)) if date_el else ''
            try:
                parsed_date = pd.to_datetime(raw_date).strftime('%Y-%m-%d')
            except Exception:
                parsed_date = ''

            reviews.append({
                'primary_key'   : str(uuid.uuid4())[:16].upper().replace('-', ''),
                'title'         : title,
                'review_text'   : review_text,
                'date'          : parsed_date,
                'rating'        : rating,
                'source'        : SOURCE_PLATFORM,
                'reviewer_name' : reviewer_name,
                'filter_used'   : filter_name,
            })
        except Exception as e:
            print(f'[WARN] Skipping card: {e}')

    return reviews

In [ ]:
# Main Scraper: Async Playwright
async def scrape_bestbuy_reviews(url: str, max_pages_per_filter: int = 10) -> list:
    """
    Asynchronously scrape BestBuy Canada product reviews across multiple
    sort filters, handling lazy-loading and pagination.

    Strategy:
        1. Launch headless Chromium with stealth headers.
        2. Navigate to the product page and scroll to reviews section.
        3. For each sort filter:
           a. Select the filter from the dropdown.
           b. Wait for review cards to re-render.
           c. Repeatedly click 'Load More' until exhausted or page limit hit.
           d. Parse all visible review HTML.
        4. Return deduplicated review list.

    Args:
        url                  : Full product page URL.
        max_pages_per_filter : Safety cap on pagination clicks per filter.

    Returns:
        List of raw review dicts.
    """
    all_reviews = []

    async with async_playwright() as pw:
        # Browser Launch with stealth configuration
        browser = await pw.chromium.launch(
            headless=True,
            args=[
                '--no-sandbox',
                '--disable-blink-features=AutomationControlled',  # Hides webdriver flag
                '--disable-extensions',
                '--disable-gpu',
            ]
        )

        context = await browser.new_context(
            # Randomise viewport to avoid fingerprinting
            viewport={'width': random.choice([1366, 1440, 1920]),
                      'height': random.choice([768, 900, 1080])},
            # Realistic desktop user agent
            user_agent=(
                'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                'AppleWebKit/537.36 (KHTML, like Gecko) '
                'Chrome/128.0.0.0 Safari/537.36'
            ),
            locale='en-CA',
            timezone_id='America/Toronto',
            # Accept Canadian cookies automatically
            extra_http_headers={
                'Accept-Language': 'en-CA,en;q=0.9',
                'Accept-Encoding': 'gzip, deflate, br',
                'DNT': '1',  # Do Not Track
            }
        )

        # Inject JS to overwrite navigator.webdriver — key stealth measure
        await context.add_init_script(
            "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
        )

        page = await context.new_page()

        # Navigate to Product Page
        print(f'Navigating to: {url}')
        await page.goto(url, wait_until='domcontentloaded', timeout=60_000)
        await asyncio.sleep(random.uniform(2, 4))  # Allow JS to settle

        # Accept cookie consent banner if present
        try:
            await page.click('button[id*="onetrust-accept"]', timeout=5_000)
            print('Cookie banner dismissed.')
        except PWTimeoutError:
            pass

        # Scroll down to reviews section to trigger lazy loading
        await page.evaluate('window.scrollBy(0, document.body.scrollHeight * 0.6)')
        await asyncio.sleep(2)

        # Iterate Over Sort Filters
        for filter_name in SORT_FILTERS:
            print(f'\n-- Applying filter: "{filter_name}" --')

            try:
                sort_selector = 'select[data-automation="sort-select"], [aria-label*="Sort"]'
                await page.wait_for_selector(sort_selector, timeout=10_000)
                await page.click(sort_selector)
                await asyncio.sleep(0.8)
                await page.select_option(sort_selector, label=filter_name)
                await asyncio.sleep(random.uniform(2, 3))

            except PWTimeoutError:
                try:
                    await page.click(f'text="{filter_name}"')
                    await asyncio.sleep(2)
                except Exception:
                    print(f'  [WARN] Could not apply filter "{filter_name}" -- skipping.')
                    continue

            # Pagination: Click 'Load More' repeatedly
            for page_num in range(max_pages_per_filter):
                html = await page.content()
                page_reviews = parse_reviews_from_html(html, filter_name)
                all_reviews.extend(page_reviews)
                print(f'  Page {page_num + 1}: +{len(page_reviews)} reviews '
                      f'(total so far: {len(all_reviews)})')

                load_more_sel = (
                    '[data-automation="show-more-reviews"], '
                    'button:has-text("Load More"), '
                    'button:has-text("Show More")'
                )
                try:
                    load_btn = page.locator(load_more_sel).first
                    if await load_btn.is_visible(timeout=4_000):
                        await load_btn.scroll_into_view_if_needed()
                        await asyncio.sleep(random.uniform(0.5, 1.5))
                        await load_btn.click()
                        await asyncio.sleep(random.uniform(2, 3))
                    else:
                        print('  No more pages for this filter.')
                        break
                except PWTimeoutError:
                    print('  Load More button not found -- end of reviews.')
                    break

        await browser.close()

    return all_reviews


# Run the Scraper
print('Starting BestBuy Canada scraper...')
raw_reviews = await scrape_bestbuy_reviews(PRODUCT_URL, max_pages_per_filter=15)
print(f'\nTotal raw reviews collected: {len(raw_reviews)}')

## Section 3 — Data Cleaning & Deduplication

In [ ]:
# Build DataFrame
df_raw = pd.DataFrame(raw_reviews)
print(f'Raw shape: {df_raw.shape}')
df_raw.head(3)

In [ ]:
# Deduplication
# Same review may appear under multiple filters
# Deduplicate on (title + reviewer_name + date)
before = len(df_raw)
df = df_raw.drop_duplicates(subset=['title', 'reviewer_name', 'date'])
print(f'Removed {before - len(df)} duplicates. Unique reviews: {len(df)}')

# Reassign primary keys to be clean UUIDs after dedup
df = df.copy()
df['primary_key'] = ['R' + str(uuid.uuid4())[:13].upper().replace('-', '') for _ in range(len(df))]

# Type Coercions
df['rating']  = pd.to_numeric(df['rating'], errors='coerce')
df['date']    = pd.to_datetime(df['date'], errors='coerce').dt.strftime('%Y-%m-%d')
df['source']  = SOURCE_PLATFORM

# Combined Text Field for NLP
# Concatenate title + body for richer sentiment signal
df['full_text'] = df['title'].fillna('') + ' ' + df['review_text'].fillna('')
df['full_text'] = df['full_text'].str.strip()

# Handle Missing Values
df['rating']        = df['rating'].fillna(df['rating'].median())
df['reviewer_name'] = df['reviewer_name'].fillna('Anonymous')
df['title']         = df['title'].fillna('(No Title)')
df['review_text']   = df['review_text'].fillna('')

print('\nCleaned dataset info:')
df.info()

In [ ]:
# Export Raw Cleaned CSV
df.to_csv(OUTPUT_CSV, index=False)
print(f'Saved raw cleaned data to: {OUTPUT_CSV}')
df[['primary_key','title','reviewer_name','date','rating','source']].head(10)

## Section 4 — Sentiment Analysis

### Two-Stage Pipeline

**Stage 1 — VADER (Valence Aware Dictionary and sEntiment Reasoner)**  
- Rule-based lexicon; fast; handles social media / review language well.  
- Produces a compound score in [-1, +1].  
- Polarity thresholds: Positive >= 0.05, Negative <= -0.05, else Neutral.  

**Stage 2 — Zero-Shot Classification (facebook/bart-large-mnli)**  
- Classifies each review into one or more aspect categories without task-specific training.  
- Candidate labels cover functional product aspects (Design, Battery, Connectivity, etc.).  

In [ ]:
# Stage 1: VADER Sentiment Scoring
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

def get_vader_sentiment(text: str):
    """
    Score text using VADER and return (compound_score, polarity_label).

    Thresholds (recommended by VADER authors):
        compound >= 0.05  -> Positive
        compound <= -0.05 -> Negative
        otherwise         -> Neutral
    """
    if not text.strip():
        return 0.0, 'Neutral'

    scores = analyzer.polarity_scores(text)
    compound = scores['compound']

    if compound >= 0.05:
        polarity = 'Positive'
    elif compound <= -0.05:
        polarity = 'Negative'
    else:
        polarity = 'Neutral'

    return compound, polarity


# Apply VADER to every review (typically < 5 seconds for 500 reviews)
tqdm.pandas(desc='VADER scoring')
df[['sentiment_score', 'sentiment_polarity']] = (
    df['full_text']
    .progress_apply(lambda t: pd.Series(get_vader_sentiment(t)))
)

print('\nSentiment polarity distribution:')
print(df['sentiment_polarity'].value_counts())

In [ ]:
# Stage 2: Aspect-Based Categorisation (Zero-Shot)
# We use facebook/bart-large-mnli (runs on CPU, approx 1.6GB download)
# For each review we identify the MOST LIKELY aspect (product feature area)

from transformers import pipeline

print('Loading zero-shot classifier (may take ~60s on first run)...')
classifier = pipeline(
    'zero-shot-classification',
    model='facebook/bart-large-mnli',
    device=-1  # CPU
)
print('Classifier loaded.')

# Aspect candidate labels covering product feature areas
ASPECT_CANDIDATES = [
    'Sound Quality',
    'Noise Cancellation',
    'Battery Life',
    'Comfort & Fit',
    'Design & Build Quality',
    'Connectivity & Pairing',
    'App & Software Features',
    'Value for Money',
    'Call Quality & Microphone',
    'Shipping & Packaging',
]

In [ ]:
# Aspect Classification Helper
def classify_aspect(text: str, threshold: float = 0.25) -> list:
    """
    Use zero-shot classification to assign aspect labels to a review.

    Returns all labels with confidence > threshold (multi-label).
    Falls back to top-1 label if no label exceeds threshold.

    Args:
        text      : Combined review title + body.
        threshold : Minimum confidence to include a label.

    Returns:
        List of aspect strings (may contain multiple aspects per review).
    """
    if not text.strip() or len(text) < 10:
        return ['Uncategorised']

    try:
        result = classifier(
            text[:512],  # Truncate to model max length
            candidate_labels=ASPECT_CANDIDATES,
            multi_label=True,
        )
        selected = [
            label for label, score in zip(result['labels'], result['scores'])
            if score >= threshold
        ]
        return selected if selected else [result['labels'][0]]
    except Exception as e:
        print(f'[WARN] Classification error: {e}')
        return ['Uncategorised']

In [ ]:
# Run Aspect Classification
# NOTE: This takes ~3-5 minutes for 200 reviews on CPU
# For faster execution, reduce the dataset or use a lighter model

print('Running aspect classification...')
df['aspects'] = [
    classify_aspect(text)
    for text in tqdm(df['full_text'], desc='Aspect tagging')
]
print('Done.')

## Section 5 — Sentiment Categorisation Tags

Combine **aspect** + **polarity** into the required tag format:  
> `['Sound Quality (Pos)']`, `['Battery Life (Neg)']`, `['Value for Money (Neu)']`

In [ ]:
# Generate Sentiment Category Tags
POLARITY_SUFFIX = {'Positive': 'Pos', 'Negative': 'Neg', 'Neutral': 'Neu'}

def build_sentiment_tags(aspects: list, polarity: str) -> list:
    """
    Combine aspect labels with polarity abbreviation into formatted tags.

    Example:
        aspects  = ['Sound Quality', 'Comfort & Fit']
        polarity = 'Positive'
        Result   = ['Sound Quality (Pos)', 'Comfort & Fit (Pos)']
    """
    suffix = POLARITY_SUFFIX.get(polarity, 'Neu')
    return [f'{aspect} ({suffix})' for aspect in aspects]


df['sentiment_tags'] = df.apply(
    lambda row: build_sentiment_tags(row['aspects'], row['sentiment_polarity']),
    axis=1
)

# String representation for CSV export
df['sentiment_tags_str'] = df['sentiment_tags'].apply(str)

print('Sample sentiment tags:')
df[['title', 'sentiment_polarity', 'sentiment_tags']].head(10)

## Section 6 — Business Insights & Visualisations

In [ ]:
# Imports for Visualisation
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

sns.set_theme(style='darkgrid', palette='viridis')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

In [ ]:
# Plot 1: Key Distributions
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('BestBuy Reviews -- Key Distributions', fontsize=14, fontweight='bold')

# Rating histogram
sns.countplot(data=df, x='rating', palette='Blues_d', ax=axes[0])
axes[0].set_title('Star Rating Distribution')
axes[0].set_xlabel('Rating (out of 5)')
axes[0].set_ylabel('Count')

# Sentiment polarity pie chart
pol_counts = df['sentiment_polarity'].value_counts()
colors = ['#2ecc71', '#e74c3c', '#95a5a6']
axes[1].pie(pol_counts, labels=pol_counts.index, colors=colors,
            autopct='%1.1f%%', startangle=140, textprops={'fontsize': 10})
axes[1].set_title('Sentiment Polarity')

# Aspect tag frequency
all_aspects = [asp for aspects_list in df['aspects'] for asp in aspects_list]
aspect_counts = Counter(all_aspects).most_common(8)
asp_labels, asp_vals = zip(*aspect_counts)
axes[2].barh(asp_labels, asp_vals, color=sns.color_palette('viridis', len(asp_labels)))
axes[2].set_title('Top Mentioned Aspects')
axes[2].set_xlabel('Frequency')
axes[2].invert_yaxis()

plt.tight_layout()
plt.savefig('task1_distributions.png', bbox_inches='tight')
plt.show()
print('Saved: task1_distributions.png')

In [ ]:
# Plot 2: Sentiment Score vs Rating (Correlation Check)
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=df, x='rating', y='sentiment_score', palette='RdYlGn', ax=ax)
ax.set_title('VADER Sentiment Score by Star Rating', fontweight='bold')
ax.set_xlabel('Star Rating')
ax.set_ylabel('VADER Compound Score')
ax.axhline(0, color='grey', linestyle='--', linewidth=1)
plt.tight_layout()
plt.savefig('task1_sentiment_vs_rating.png', bbox_inches='tight')
plt.show()

In [ ]:
# Plot 3: Reviews Over Time
df['date_parsed'] = pd.to_datetime(df['date'], errors='coerce')
monthly = df.set_index('date_parsed').resample('ME').size().reset_index()
monthly.columns = ['month', 'count']

fig, ax = plt.subplots(figsize=(10, 4))
ax.fill_between(monthly['month'], monthly['count'], alpha=0.3, color='steelblue')
ax.plot(monthly['month'], monthly['count'], color='steelblue', linewidth=2)
ax.set_title('Reviews Posted per Month', fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Number of Reviews')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('task1_reviews_over_time.png', bbox_inches='tight')
plt.show()

In [ ]:
# Business Insight Summary
avg_rating   = df['rating'].mean()
pct_positive = (df['sentiment_polarity'] == 'Positive').mean() * 100
pct_negative = (df['sentiment_polarity'] == 'Negative').mean() * 100

top_positive_asp = [a for a, c in Counter(
    [asp for _, asps in df[df['sentiment_polarity']=='Positive']['aspects'].items()
     for asp in asps]).most_common(3)]
top_negative_asp = [a for a, c in Counter(
    [asp for _, asps in df[df['sentiment_polarity']=='Negative']['aspects'].items()
     for asp in asps]).most_common(3)]

print('=' * 60)
print(' BUSINESS INSIGHTS SUMMARY')
print('=' * 60)
print(f' Product        : Sony WH-1000XM5 Headphones')
print(f' Total Reviews  : {len(df)}')
print(f' Avg Rating     : {avg_rating:.2f} / 5.0')
print(f' % Positive     : {pct_positive:.1f}%')
print(f' % Negative     : {pct_negative:.1f}%')
print(f'\n Satisfaction Drivers (Positive reviews):')
for asp in top_positive_asp:
    print(f'   [+] {asp}')
print(f'\n Pain Points (Negative reviews):')
for asp in top_negative_asp:
    print(f'   [-] {asp}')
print('=' * 60)

In [ ]:
# Export Final Dataset with Sentiment
export_cols = [
    'primary_key', 'title', 'review_text', 'date', 'rating',
    'source', 'reviewer_name', 'sentiment_score',
    'sentiment_polarity', 'sentiment_tags_str'
]

df_export = df[export_cols].copy()

# CSV export
df_export.to_csv('bestbuy_reviews_sentiment.csv', index=False)

# Excel export with formatted headers
with pd.ExcelWriter(OUTPUT_XLSX, engine='openpyxl') as writer:
    df_export.to_excel(writer, sheet_name='Reviews', index=False)

print(f'Exported {len(df_export)} reviews to:')
print(f'  - bestbuy_reviews_sentiment.csv')
print(f'  - {OUTPUT_XLSX}')

## Section 7 — Anti-Scraping Mitigation Strategies

### 7.1 IP Blocking

**Problem:** Websites track request volume per IP and block IPs exceeding thresholds.

**Solutions Implemented:**
- **Rotating Proxies:** Assign a different proxy IP to each browser context. Residential IPs (Bright Data, Oxylabs) are indistinguishable from real users.
- **Request Throttling:** `human_delay()` introduces 1.5-3.5 second pauses between actions.
- **Session Rotation:** Create a new browser context every N requests to refresh IP + cookie state.

```python
# Example: rotating proxy configuration in Playwright
PROXY_LIST = ['http://proxy1:port', 'http://proxy2:port']
context = await browser.new_context(
    proxy={'server': random.choice(PROXY_LIST)}
)
```

---

### 7.2 CAPTCHA & Bot Detection

**Problem:** Sites use CAPTCHA challenges and headless browser detection (Cloudflare, PerimeterX).

**Solutions:**
- **navigator.webdriver removal:** JS injection hides the automation flag.
- **User-Agent rotation:** Different realistic UA string per session.
- **Realistic headers:** `Accept-Language`, `DNT`, `Referer` mimic real browsers.
- **Anti-CAPTCHA APIs:** 2Captcha or AntiCaptcha solve reCAPTCHA v2/v3 programmatically.
- **Mouse simulation:** `page.mouse.move()` with natural curves mimics human pointer.

```python
# Stealth: overwrite webdriver fingerprint
await context.add_init_script(
    "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
)
```

---

### 7.3 Retry Logic with Exponential Backoff

**Problem:** Transient network errors and rate-limit (HTTP 429) responses cause failures.

```python
async def fetch_with_retry(page, url: str, max_retries: int = 5):
    for attempt in range(max_retries):
        try:
            await page.goto(url, wait_until='domcontentloaded', timeout=30_000)
            return  # Success
        except Exception as e:
            wait_time = 2 ** attempt  # Exponential backoff: 2, 4, 8, 16, 32 seconds
            print(f'Attempt {attempt+1} failed. Retrying in {wait_time}s...')
            await asyncio.sleep(wait_time)
    raise RuntimeError(f'Failed after {max_retries} attempts: {url}')
```

---

### 7.4 Summary Table

| Challenge | Solution Applied |
|---|---|
| IP Blocking | Rotating residential proxies + request throttling |
| CAPTCHA | Anti-CAPTCHA API + stealth browser mode |
| Bot Detection | Remove `navigator.webdriver`, rotate User-Agent, randomise timing |
| Rate Limiting | Exponential backoff retry + session rotation |
| JavaScript Rendering | Playwright headless browser (vs. raw HTTP requests) |